# 04 SVHN DNN：Simple Dense DNN

本 notebook 接在 `03_svhn_svm_baseline.ipynb` 後面。

前一份已顯示：同樣是 0-9 數字分類，SVHN 比 `load_digits` 困難許多，`raw-pixel` 傳統 ML 的表現明顯受限。

本階段使用 TensorFlow/Keras 建立第一個深度學習模型：**Simple Dense DNN**。模型會先將 `32x32x3` 圖片攤平成 3072 維向量，再透過三層全連接神經網路進行分類。

這份模型刻意保持單純：`Flatten -> Dense(512) -> Dense(256) -> Dense(128) -> softmax`。它用來建立深度學習基本流程，並作為下一份 `05_svhn_dnn_optimization.ipynb` 進階優化的比較基準。

> 注意：本階段尚未進入 CNN。Simple Dense DNN 雖然是神經網路，但圖片一開始仍被 flatten，因此尚未利用影像的空間結構。


## 0. 匯入套件與設定

這份 notebook 和 `03` 使用相同的資料抽樣設定，方便比較。


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report

import tensorflow as tf

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

TRAIN_SAMPLES = 12000
TEST_SAMPLES = 3000

EPOCHS = 50
BATCH_SIZE = 128


## 1. 下載 SVHN cropped

若你剛剛已經跑過 `03`，Colab 會使用快取，不會重新下載完整資料。


In [ ]:
SVHN_BASE_URL = 'http://ufldl.stanford.edu/housenumbers/'

train_path = tf.keras.utils.get_file(
    'svhn_train_32x32.mat',
    origin=SVHN_BASE_URL + 'train_32x32.mat',
    cache_dir='.',
    cache_subdir='data/svhn'
)

test_path = tf.keras.utils.get_file(
    'svhn_test_32x32.mat',
    origin=SVHN_BASE_URL + 'test_32x32.mat',
    cache_dir='.',
    cache_subdir='data/svhn'
)

print('train_path:', train_path)
print('test_path:', test_path)


## 2. 讀取資料與固定抽樣

抽樣邏輯和 `03_svhn_svm_baseline.ipynb` 相同。


In [ ]:
def load_svhn_mat(path):
    data = loadmat(path)
    X = data['X']
    y = data['y'].reshape(-1)

    # 原始 shape: (height, width, channel, sample) -> (sample, height, width, channel)
    X = np.transpose(X, (3, 0, 1, 2))

    # SVHN 原始 label: 1-9 是原本數字，10 代表 0。
    y = y % 10
    return X, y.astype('int64')

def stratified_subset(X, y, n_samples, random_state=42):
    rng = np.random.default_rng(random_state)
    classes = np.unique(y)
    base = n_samples // len(classes)
    remainder = n_samples % len(classes)

    selected = []
    for rank, cls in enumerate(classes):
        cls_idx = np.where(y == cls)[0]
        take = base + (1 if rank < remainder else 0)
        take = min(take, len(cls_idx))
        selected.append(rng.choice(cls_idx, size=take, replace=False))

    selected = np.concatenate(selected)
    rng.shuffle(selected)
    return X[selected], y[selected]

X_train_full, y_train_full = load_svhn_mat(train_path)
X_test_full, y_test_full = load_svhn_mat(test_path)

X_train, y_train = stratified_subset(X_train_full, y_train_full, TRAIN_SAMPLES, RANDOM_STATE)
X_test, y_test = stratified_subset(X_test_full, y_test_full, TEST_SAMPLES, RANDOM_STATE + 1)

print('X_train:', X_train.shape, X_train.dtype)
print('X_test:', X_test.shape, X_test.dtype)
print('train class counts:', dict(zip(*np.unique(y_train, return_counts=True))))
print('test class counts:', dict(zip(*np.unique(y_test, return_counts=True))))


## 3. 觀察資料與正規化

SVHN 是 0-255 RGB 影像，因此神經網路訓練前先除以 255。


In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(12, 4))
for ax, image, label in zip(axes.ravel(), X_train[:30], y_train[:30]):
    ax.imshow(image)
    ax.set_title(str(label))
    ax.axis('off')
plt.suptitle('SVHN examples: real-world street-view digits')
plt.tight_layout()
plt.show()

print('pixel min:', X_train.min())
print('pixel max:', X_train.max())
print('SVHN 是一般 8-bit RGB 影像，正規化到 0~1 時除以 255.0。')


In [ ]:
X_train_float = X_train.astype('float32') / 255.0
X_test_float = X_test.astype('float32') / 255.0

print('X_train_float:', X_train_float.shape, X_train_float.min(), X_train_float.max())
print('X_test_float:', X_test_float.shape, X_test_float.min(), X_test_float.max())


## 4. 建立 Simple Dense DNN

這個模型會先用 `Flatten()` 把 `32x32x3` 圖片攤平成 3072 維，再接三層 Dense layers。

概念對照：

| Keras 元件 | 概念 |
| --- | --- |
| `Flatten` | 把影像轉成 feature vector |
| `Dense` | 全連接層，學習輸入特徵與類別之間的非線性關係 |
| `ReLU` | 常用 activation function，讓模型能學習非線性關係 |
| `softmax` | 輸出 10 個數字類別機率 |
| `sparse_categorical_crossentropy` | 多類別分類 loss |

這份 Simple Dense DNN 不使用 Dropout 與 Batch Normalization，目的是先建立一個清楚、容易理解的 Dense network。下一份 notebook 會在此基礎上加入 LeakyReLU、batch size 調整與 learning rate schedule，觀察是否能進一步提升準確率。


In [ ]:
def build_simple_dense_dnn():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

tf.keras.backend.clear_session()
tf.random.set_seed(RANDOM_STATE)

model = build_simple_dense_dnn()
model.summary()


## 5. 訓練 Simple Dense DNN

本階段使用 EarlyStopping 監控 validation accuracy。若 validation accuracy 連續多次沒有改善，訓練會提前停止，並回復到 validation 表現最好的權重。


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=8,
    restore_best_weights=True
)

start = time.perf_counter()
history = model.fit(
    X_train_float,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)
training_time = time.perf_counter() - start
print('training time:', training_time)


## 6. 觀察訓練曲線

本節呈現的重點如下：神經網路不是只看最後 accuracy，也要看 train/validation loss 是否分開。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train loss')
axes[0].plot(history.history['val_loss'], label='val loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='train accuracy')
axes[1].plot(history.history['val_accuracy'], label='val accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('epoch')
axes[1].legend()

plt.tight_layout()
plt.show()


## 7. 評估 Simple Dense DNN 結果

本節以表格方式整理 Simple Dense DNN 的主要結果，包括訓練分數、測試分數、最佳 validation accuracy、實際訓練 epoch 數與訓練時間。後續 `05` 會使用同一份 SVHN 子集與相同評估方式，觀察進階優化是否真的改善。


In [ ]:
train_loss, train_acc = model.evaluate(X_train_float, y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_test_float, y_test, verbose=0)

results_df = pd.DataFrame([
    {
        'model': 'Simple Dense DNN',
        'train_acc': train_acc,
        'test_acc': test_acc,
        'best_val_acc': max(history.history['val_accuracy']),
        'epochs_ran': len(history.history['loss']),
        'training_time_sec': training_time,
    }
])
display(results_df)

proba = model.predict(X_test_float, verbose=0)
y_pred = proba.argmax(axis=1)
print(classification_report(y_test, y_pred))


## 7.1 目前模型比較：SVM 與 Simple Dense DNN

以下表格使用同一份 SVHN cropped 子集進行比較：train 12,000 筆、test 3,000 筆。分數是課程目前固定的實測參考值；若重新執行 notebook，因硬體、TensorFlow 版本或隨機性，數值可能有小幅差異。

| 方法 | Train Accuracy | Test Accuracy | 觀察 |
| --- | ---: | ---: | --- |
| RBF SVM raw pixels + StandardScaler | 0.6947 | 0.5170 | 傳統 ML 可用，但真實影像 raw pixels 的測試表現有限。 |
| Simple Dense DNN | 0.7476 | 0.6540 | 神經網路能學到更複雜的非線性關係，測試準確率明顯提升。 |

從這個比較可以看到，Simple Dense DNN 已經比 RBF SVM 更適合處理 SVHN 這類真實影像資料。不過它仍然把圖片攤平成一維向量，因此下一步會嘗試 DNN 訓練策略優化，再進一步銜接 CNN。


## 8. 錯誤分析

透過錯誤案例觀察 Simple Dense DNN 的限制。它雖然是神經網路，但一開始就把圖片攤平，沒有明確利用上下左右像素的空間關係。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues', colorbar=False)
plt.title('Simple Dense DNN confusion matrix')
plt.show()

wrong_idx = np.where(y_pred != y_test)[0]
print('wrong predictions:', len(wrong_idx))

n_show = min(12, len(wrong_idx))
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for ax, idx in zip(axes.ravel(), wrong_idx[:n_show]):
    ax.imshow(X_test[idx])
    confidence = proba[idx, y_pred[idx]]
    ax.set_title(f't={y_test[idx]} p={y_pred[idx]}\nconf={confidence:.2f}')
    ax.axis('off')
for ax in axes.ravel()[n_show:]:
    ax.axis('off')
plt.tight_layout()
plt.show()


## 9. 小結：下一步如何優化 DNN？

Simple Dense DNN 已經比 raw-pixel 傳統 ML 更能處理 SVHN，但它仍然把圖片攤平成一維向量，沒有直接利用影像的上下左右空間關係。

下一份 `05_svhn_dnn_optimization.ipynb` 會保留相同的 Dense layer 寬度，直接加入一組進階優化策略：

- 將 ReLU 改成 LeakyReLU。
- 將 batch size 從 128 改成 64。
- 加入 `ReduceLROnPlateau`，當 validation accuracy 停滯時自動降低 learning rate。
- 使用較長訓練上限與 EarlyStopping，讓模型有機會繼續改善但避免無止境訓練。

觀察重點是：這些訓練策略是否能讓 test accuracy 從 Simple Dense DNN 的水準進一步提升到接近 70%。


## 課後練習與思考題

請在目前的 Simple Dense DNN 之後，嘗試從模型架構與訓練參數進行調整。練習時請同時觀察 train accuracy 與 test accuracy，避免只看單一分數。

### 練習 1：調整 DNN 寬度

調整隱藏層寬度，比較較小與較大的 Dense 架構。觀察較寬的模型是否一定能改善 `test_acc`。

In [ ]:
# 練習 1 參考答案：調整 DNN 寬度

def build_width_variant(hidden_units=(512, 256, 128), learning_rate=0.001):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(32, 32, 3)))
    model.add(tf.keras.layers.Flatten())
    for units in hidden_units:
        model.add(tf.keras.layers.Dense(units, activation='relu'))
    model.add(tf.keras.layers.Dense(10, activation='softmax'))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

width_experiments = [
    ('narrow', (256, 128, 64)),
    ('wide', (1024, 512, 256)),
]

rows = []
practice_epochs = 5
for name, hidden_units in width_experiments:
    tf.keras.backend.clear_session()
    practice_model = build_width_variant(hidden_units=hidden_units)
    start = time.perf_counter()
    practice_history = practice_model.fit(
        X_train_float,
        y_train,
        validation_split=0.2,
        epochs=practice_epochs,
        batch_size=BATCH_SIZE,
        verbose=0
    )
    elapsed = time.perf_counter() - start
    train_loss, train_acc = practice_model.evaluate(X_train_float, y_train, verbose=0)
    test_loss, test_acc = practice_model.evaluate(X_test_float, y_test, verbose=0)
    rows.append({
        'variant': name,
        'hidden_units': str(hidden_units),
        'train_acc': train_acc,
        'test_acc': test_acc,
        'best_val_acc': max(practice_history.history['val_accuracy']),
        'training_time_sec': elapsed,
    })

width_df = pd.DataFrame(rows).sort_values('test_acc', ascending=False)
display(width_df)


#### 練習 1 參考答案：調整 DNN 寬度

較寬的模型容量更大，但不保證 `test_acc` 一定同步提升。判讀時應同時看 `train_acc` 與 `test_acc`，而不是只看訓練分數。


### 練習 2：調整 DNN 深度

調整隱藏層深度，觀察加深後是否一定能提升測試準確率。請特別比較 `train_acc` 與 `test_acc` 是否同步提升。

In [ ]:
# 練習 2 參考答案：調整 DNN 深度

depth_experiments = [
    ('3 dense layers', (512, 256, 128)),
    ('4 dense layers', (512, 256, 128, 64)),
]

rows = []
practice_epochs = 5
for name, hidden_units in depth_experiments:
    tf.keras.backend.clear_session()
    practice_model = build_width_variant(hidden_units=hidden_units)
    start = time.perf_counter()
    practice_history = practice_model.fit(
        X_train_float,
        y_train,
        validation_split=0.2,
        epochs=practice_epochs,
        batch_size=BATCH_SIZE,
        verbose=0
    )
    elapsed = time.perf_counter() - start
    train_loss, train_acc = practice_model.evaluate(X_train_float, y_train, verbose=0)
    test_loss, test_acc = practice_model.evaluate(X_test_float, y_test, verbose=0)
    rows.append({
        'variant': name,
        'hidden_units': str(hidden_units),
        'train_acc': train_acc,
        'test_acc': test_acc,
        'best_val_acc': max(practice_history.history['val_accuracy']),
        'training_time_sec': elapsed,
    })

depth_df = pd.DataFrame(rows).sort_values('test_acc', ascending=False)
display(depth_df)


#### 練習 2 參考答案：調整 DNN 深度

- 觀察說明：增加深度可能提升表達能力，也可能讓訓練更難或更容易過擬合。
- 若 train_acc 提升但 test_acc 沒有提升，代表更深的模型未必更適合目前資料與訓練設定。


### 練習 3：調整 learning rate

調整 learning rate，觀察學習速度與穩定性的差異。請比較不同 learning rate 的 validation 表現與最後測試分數。

In [ ]:
# 練習 3 參考答案：調整 learning rate

learning_rate_experiments = [0.001, 0.0003, 0.0001]
rows = []
practice_epochs = 5

for lr in learning_rate_experiments:
    tf.keras.backend.clear_session()
    practice_model = build_width_variant(hidden_units=(512, 256, 128), learning_rate=lr)
    start = time.perf_counter()
    practice_history = practice_model.fit(
        X_train_float,
        y_train,
        validation_split=0.2,
        epochs=practice_epochs,
        batch_size=BATCH_SIZE,
        verbose=0
    )
    elapsed = time.perf_counter() - start
    train_loss, train_acc = practice_model.evaluate(X_train_float, y_train, verbose=0)
    test_loss, test_acc = practice_model.evaluate(X_test_float, y_test, verbose=0)
    rows.append({
        'learning_rate': lr,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'best_val_acc': max(practice_history.history['val_accuracy']),
        'final_val_loss': practice_history.history['val_loss'][-1],
        'training_time_sec': elapsed,
    })

lr_df = pd.DataFrame(rows).sort_values('test_acc', ascending=False)
display(lr_df)


#### 練習 3 參考答案：調整 learning rate

Learning rate 太大可能讓訓練震盪，太小可能讓模型學得太慢。實務上常搭配 validation score、learning curve 與 learning rate scheduler 一起判斷。


### 練習 4：整理 Dense DNN 的限制

思考為什麼 Simple Dense DNN 變深或變寬後，仍然不一定能有效處理真實影像。

#### 練習 4 參考答案：整理 Dense DNN 的限制

| 限制 | 說明 |
| --- | --- |
| 空間結構流失 | `Flatten` 會把 `32x32x3` 圖片攤平成 3072 維向量，模型不再直接知道相鄰像素之間的位置關係。 |
| 參數效率較差 | Dense layer 需要為大量像素連線學權重，對影像局部特徵的利用不如卷積有效率。 |
| 更容易受背景干擾 | SVHN 有背景、裁切與光線變化，Dense DNN 缺少專門擷取局部邊緣與形狀的機制。 |
| 變深變寬不是萬能 | 模型容量增加可能提升 `train_acc`，但若沒有更好的影像歸納偏置，`test_acc` 仍可能卡住。 |

Dense DNN 可以展現深度學習相對於傳統 ML 的改善，但影像辨識主流仍會進一步使用 CNN。
